# Portfolio Optimization Using Modern Portfolio Theory

### Real Equity Positions — Fidelity Brokerage Accounts

This notebook applies **Modern Portfolio Theory (MPT)** to real equity positions stored
in MySQL via the `Portfolio_Positions` and `equity_historical` tables.

| Aspect | Value |
|--------|-------|
| Data source | MySQL cache via `portfolio_app/src/data.py` (no raw SQL) |
| Assets | 80+ real equity holdings |
| Frequency | 252 trading days/year |
| Accounts | Multiple (taxable, retirement, 529, HSA) |
| Optimization | Max Sharpe, Min Volatility, Current comparison |
| Covariance | Ledoit-Wolf shrinkage |

**Strategy Steps:**
1. Load current positions via `data.py` → aggregate by symbol
2. Build 3-year daily close price matrix from `equity_historical`
3. Calculate expected returns (geometric, 252-day annualized)
4. Estimate covariance (Ledoit-Wolf shrinkage)
5. Optimize: **Max Sharpe**, **Min Volatility**
6. Compare current vs optimal — visualize efficient frontier
7. Generate discrete (whole-share) trade allocation

> **Plan:** See `portfolio_optimization_strategy.md` in this folder  
> **Author:** Prashant Rajoria | **Date:** 2026-02-26 | **Branch:** `openbb_learning`

## 1. Setup & Imports

In [27]:
import sys
import os

# Windows encoding fix (skip in Jupyter where stdout is OutStream)
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pathlib import Path

# Derive paths relative to this notebook
_notebook_dir = Path(os.getcwd())
_portfolio_app = _notebook_dir.parent if _notebook_dir.name == "analysis" else _notebook_dir
project_root = _portfolio_app.parent
src_dir = _portfolio_app / "src"

# Fallback for non-standard CWD
if not (src_dir / "data.py").exists():
    src_dir = Path(r"I:\masterswork\git\OpenBB\portfolio_app\src")
    project_root = Path(r"I:\masterswork\git\OpenBB")

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Skip auto-create overhead
os.environ.setdefault("FMP_CACHE_AUTO_CREATE_DB", "false")

# Load .env for database credentials
from dotenv import load_dotenv
load_dotenv(project_root / ".env", override=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pypfopt import expected_returns, EfficientFrontier, CLA
from pypfopt import CovarianceShrinkage
from pypfopt import plotting as pplt
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

print(f"Project root: {project_root}")
print(f"Source dir:    {src_dir}")
print("All imports successful ✓")

Project root: i:\masterswork\git\OpenBB
Source dir:    i:\masterswork\git\OpenBB\portfolio_app\src
All imports successful ✓


In [28]:
# Import data and service layers
from data import (
    get_positions_df,
    get_equity_historical_df,
    get_latest_prices_df,
    get_distinct_symbols,
    get_distinct_accounts,
    check_db,
)
import service

# Verify DB connection
db_ok = check_db()
print(f"Database connected: {db_ok}")
if not db_ok:
    raise RuntimeError("Cannot connect to database — check .env credentials")

Database connected: True


## 2. Load Current Positions

Fetch positions via `data.get_positions_df()` (latest snapshot, joined with `Account_Owner`).  
Each row is a cost-basis lot — we aggregate by symbol to get portfolio-wide totals.

In [29]:
# Load positions (latest snapshot, all lots)
positions_raw = get_positions_df()
print(f"Raw lots: {len(positions_raw)}")
print(f"Unique symbols: {positions_raw['symbol'].nunique()}")
print(f"Accounts: {positions_raw['account_name'].nunique()}")
print(f"Snapshot: {positions_raw['snapshot_date'].iloc[0]}")

# Filter out cash and zero-quantity positions
positions = positions_raw[
    (~positions_raw["symbol"].isin(["Cash", "Pending Activity"]))
    & (positions_raw["quantity"] > 0)
].copy()
print(f"Equity lots (after filtering): {len(positions)}")

Raw lots: 584
Unique symbols: 93
Accounts: 8
Snapshot: 2026-02-20T01:39:00
Equity lots (after filtering): 578


In [30]:
# Aggregate lots by symbol across all accounts
portfolio = (
    positions
    .groupby("symbol")
    .agg(
        description=("description", "first"),
        total_qty=("quantity", "sum"),
        total_value=("current_value", "sum"),
        total_cost=("cost_basis_total", "sum"),
        total_gl=("total_gain_loss", "sum"),
    )
    .sort_values("total_value", ascending=False)
)

portfolio["weight_pct"] = portfolio["total_value"] / portfolio["total_value"].sum() * 100
portfolio["gl_pct"] = np.where(
    portfolio["total_cost"] != 0,
    portfolio["total_gl"] / portfolio["total_cost"] * 100,
    0,
)

# Symbol → company name lookup (used throughout this notebook)
sym_name = portfolio["description"].to_dict()

total_portfolio_value = portfolio["total_value"].sum()
print(f"Total portfolio value: ${total_portfolio_value:,.2f}")
print(f"Unique symbols: {len(portfolio)}")
print(f"Top 10 concentration: {portfolio['weight_pct'].head(10).sum():.1f}%")
print()

print(f"{'Symbol':<8} {'Company':<35} {'Value':>12} {'Weight%':>8} {'G/L%':>8}")
print("-" * 75)
for sym, row in portfolio.head(20).iterrows():
    print(f"{sym:<8} {str(row['description'])[:35]:<35} ${row['total_value']:>10,.2f} "
          f"{row['weight_pct']:>7.2f}% {row['gl_pct']:>7.1f}%")
print(f"\n... showing top 20 of {len(portfolio)} positions")

Total portfolio value: $3,348,295.08
Unique symbols: 92
Top 10 concentration: 86.0%

Symbol   Company                                    Value  Weight%     G/L%
---------------------------------------------------------------------------
MSFT     MICROSOFT CORP                      $1,631,435.41   48.72%   119.9%
09261F598 BTC LPATH IDX 2045 M                $743,360.08   22.20%    44.0%
NVDA     NVIDIA CORPORATION COM              $160,515.25    4.79%  1301.6%
AAPL     APPLE INC                           $ 78,391.50    2.34%    92.7%
NHX202764 NH PORTFOLIO 2027 (FIDELITY INDEX)  $ 78,261.69    2.34%     5.8%
GOOGL    ALPHABET INC CAP STK CL A           $ 75,612.00    2.26%   170.6%
NHX203309 NH PORTFOLIO 2033 (FIDELITY INDEX)  $ 38,875.50    1.16%     2.3%
AMZN     AMAZON.COM INC                      $ 27,080.60    0.81%    85.3%
SHOP     SHOPIFY INC COM NPV CL A ISIN #CA82 $ 23,999.90    0.72%    60.0%
WM       WASTE MANAGEMENT INC                $ 23,149.00    0.69%    12.1%
TGT     

## 3. Visualize Current Allocation

Treemap showing position sizes with gain/loss coloring.

In [31]:
# Treemap of current allocation
treemap_df = portfolio.reset_index().copy()
treemap_df["label"] = (
    treemap_df["symbol"] + "<br>" + treemap_df["weight_pct"].round(1).astype(str) + "%"
)

fig = px.treemap(
    treemap_df.head(30),
    path=["label"],
    values="total_value",
    color="gl_pct",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="Current Portfolio Allocation (Top 30 Positions)",
)
fig.update_layout(width=900, height=600)
fig.show()

## 4. Build Price Matrix (3-Year Daily Close)

Load historical prices via `data.get_equity_historical_df()` for each symbol.  
This uses the fmp_cached provider API (gap-detect → FMP fetch → cache store).

**Quality filters:**
- Drop symbols with <60% data coverage
- Forward-fill gaps up to 5 days (holidays)
- Drop remaining NaN columns

In [32]:
from datetime import date, timedelta
import time as _time
from db import query as db_query

portfolio_symbols = list(portfolio.index)
start_date = (date.today() - timedelta(days=3*365)).isoformat()
end_date = date.today().isoformat()

print(f"Loading 3-year history for {len(portfolio_symbols)} symbols ...")
print(f"Date range: {start_date} to {end_date}\n")

# Read directly from cache (no FMP API calls — fast, uses existing data only)
placeholders = ", ".join(["%s"] * len(portfolio_symbols))
history_sql = f"""
SELECT symbol, date, open, high, low, close, volume
FROM equity_historical
WHERE symbol IN ({placeholders})
  AND date >= %s AND date <= %s
  AND close IS NOT NULL AND close > 0
ORDER BY symbol, date
"""

t0 = _time.perf_counter()
rows = db_query(history_sql, tuple(portfolio_symbols) + (start_date, end_date))
history_df = pd.DataFrame(rows) if rows else pd.DataFrame()
elapsed = _time.perf_counter() - t0

symbols_found = history_df["symbol"].nunique() if not history_df.empty else 0
missing = [s for s in portfolio_symbols if s not in set(history_df["symbol"].unique())] if not history_df.empty else portfolio_symbols

print(f"Loaded {len(history_df):,} price rows in {elapsed:.1f}s")
print(f"Symbols with history: {symbols_found} / {len(portfolio_symbols)}")
if missing:
    print(f"\n⚠ Missing symbols (no cached history):")
    for s in missing:
        print(f"  {s:<8} {sym_name.get(s, '')[:40]}")

Loading 3-year history for 92 symbols ...
Date range: 2023-02-27 to 2026-02-26

Loaded 60,621 price rows in 2.5s
Symbols with history: 86 / 92

⚠ Missing symbols (no cached history):
  09261F598 BTC LPATH IDX 2045 M
  NHX202764 NH PORTFOLIO 2027 (FIDELITY INDEX)
  NHX203309 NH PORTFOLIO 2033 (FIDELITY INDEX)
  BRKB     BERKSHIRE HATHAWAY INC COM USD0.0033 CLA
  36320A203 GALAXY NEXT GENERATION INC COM NEW
  04599D597 ASIA BROADBAND INC $0.001 NEVADA RESTRIC


In [33]:
# Pivot to wide format: date × symbol close prices
prices = (
    history_df
    .assign(date=lambda d: pd.to_datetime(d["date"]))
    .pivot(index="date", columns="symbol", values="close")
    .sort_index()
)

# Drop symbols with < 60% data coverage
MIN_COVERAGE = 0.6
coverage = prices.notna().mean()
valid_symbols = coverage[coverage >= MIN_COVERAGE].index.tolist()
dropped = coverage[coverage < MIN_COVERAGE].index.tolist()

if dropped:
    print(f"Dropped {len(dropped)} symbols with <{MIN_COVERAGE*100:.0f}% coverage: {dropped}")

prices = prices[valid_symbols].dropna(how="all")

# Forward-fill small gaps (holidays, data lags)
prices = prices.ffill(limit=5)

# Drop any remaining cols with NaN
prices = prices.dropna(axis=1)

opt_symbols = list(prices.columns)
print(f"\nClean price matrix: {prices.shape[0]} days \u00d7 {prices.shape[1]} symbols")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")
print(f"Symbols for optimization: {len(opt_symbols)}")

Dropped 5 symbols with <60% coverage: ['EADSF', 'FIG', 'MVVYF', 'NSAV', 'NXDR']

Clean price matrix: 752 days × 79 symbols
Date range: 2023-02-27 to 2026-02-25
Symbols for optimization: 79


## 5. Expected Returns & Covariance Matrix

**Expected returns** — mean historical (geometric, annualized at 252 trading days):

$$\mu_i = \left(\prod_{t=1}^{N}(1 + r_{i,t})\right)^{252/N} - 1$$

**Covariance** — Ledoit-Wolf shrinkage for stability:

$$\hat{\Sigma} = \alpha \cdot S + (1 - \alpha) \cdot F$$

where $S$ = sample covariance, $F$ = structured target, $\alpha$ = shrinkage intensity.

In [34]:
# Expected returns (geometric, 252 trading days/year)
mu = expected_returns.mean_historical_return(prices, frequency=252, compounding=True)

# Covariance matrix (Ledoit-Wolf shrinkage)
cov_matrix = CovarianceShrinkage(prices, frequency=252).ledoit_wolf()

print("Expected Annual Returns (top 10 / bottom 5):")
mu_sorted = mu.sort_values(ascending=False)
for sym in mu_sorted.head(10).index:
    name = sym_name.get(sym, "")
    print(f"  {sym:<8} {name[:30]:<30} {mu_sorted[sym]:>8.1%}")
print("  ...")
for sym in mu_sorted.tail(5).index:
    name = sym_name.get(sym, "")
    print(f"  {sym:<8} {name[:30]:<30} {mu_sorted[sym]:>8.1%}")

# Annualized volatilities
vols = np.sqrt(np.diag(cov_matrix))
print(f"\nAverage annual volatility: {np.mean(vols):.1%}")
print(f"Volatility range: {np.min(vols):.1%} — {np.max(vols):.1%}")

Expected Annual Returns (top 10 / bottom 5):
  PLTR     PALANTIR TECHNOLOGIES INC CL A   157.3%
  NVDA     NVIDIA CORPORATION COM           103.6%
  HOOD     ROBINHOOD MKTS INC COM CL A      100.7%
  GE       GE AEROSPACE COM NEW              73.1%
  META     META PLATFORMS INC CLASS A COM    57.3%
  GOOGL    ALPHABET INC CAP STK CL A         52.0%
  SMCI     SUPER MICRO COMPUTER INC COM N    49.6%
  CRWD     CROWDSTRIKE HLDGS INC CL A        44.8%
  SHOP     SHOPIFY INC COM NPV CL A ISIN     44.1%
  WMT      WALMART INC COM                   39.0%
  ...
  PTON     PELOTON INTERACTIVE INC CL A C   -31.9%
  FVRR     FIVERR INTERNATIONAL LTD COM N   -34.7%
  DDD      3D SYSTEMS CORPORATION COM USD   -41.2%
  LCID     LUCID GROUP INC COM NEW          -51.3%
  AMC      AMC ENTMT HLDGS INC CL A NEW     -75.4%

Average annual volatility: 44.8%
Volatility range: 14.6% — 96.3%


## 6. Current Portfolio Metrics

Calculate expected return, volatility, and Sharpe ratio for the **current allocation**
so we can compare against the optimized portfolios.

In [35]:
# Key parameters
RISK_FREE_RATE = 0.045   # ~4.5% T-bill rate (2025-2026)
MAX_WEIGHT = 0.25        # 25% cap per asset

# Build current weights vector (only symbols in optimization universe)
current_alloc = portfolio.loc[portfolio.index.isin(opt_symbols), "total_value"]
current_weights = current_alloc / current_alloc.sum()
current_weights = current_weights.reindex(mu.index).fillna(0)

# Current portfolio statistics
current_return = float(current_weights @ mu)
current_vol = float(np.sqrt(current_weights @ cov_matrix @ current_weights))
current_sharpe = (current_return - RISK_FREE_RATE) / current_vol

print("=" * 50)
print("  CURRENT PORTFOLIO METRICS")
print("=" * 50)
print(f"  Expected Annual Return:  {current_return:>8.2%}")
print(f"  Annual Volatility:       {current_vol:>8.2%}")
print(f"  Sharpe Ratio:            {current_sharpe:>8.2f}")
print(f"  Risk-Free Rate:          {RISK_FREE_RATE:>8.2%}")
print(f"  Symbols in universe:     {len(opt_symbols):>8}")
print("=" * 50)

  CURRENT PORTFOLIO METRICS
  Expected Annual Return:    24.56%
  Annual Volatility:         21.83%
  Sharpe Ratio:                0.92
  Risk-Free Rate:             4.50%
  Symbols in universe:           79


### Understanding the Sharpe Ratio — Intuitive Explanation

The **Sharpe Ratio** answers a simple question:

> *"How much **extra return** am I getting for each unit of **risk** I'm taking?"*

Think of it like **miles per gallon** for your portfolio:
- **Return** = how far you drove (the reward)
- **Risk** (volatility) = how much fuel you burned (the cost)
- **Risk-free rate** = the distance you'd cover just coasting downhill (what you get for zero risk, e.g. T-bills)

$$\text{Sharpe Ratio} = \frac{R_p - R_f}{\sigma_p}$$

where $R_p$ = portfolio return, $R_f$ = risk-free rate, $\sigma_p$ = portfolio standard deviation.

**Interpreting the number:**

| Sharpe Ratio | Interpretation |
|:---:|---|
| **< 0** | You're earning *less* than risk-free — you're paying to take risk |
| **0 – 0.5** | Poor — the extra return doesn't justify the volatility |
| **0.5 – 1.0** | Acceptable — decent risk-adjusted performance |
| **1.0 – 2.0** | Good — strong return per unit of risk |
| **> 2.0** | Excellent — rarely sustained over long periods |

**Real-world analogy:**  
Imagine two restaurants. Both serve $20 meals. Restaurant A has consistent quality (low volatility) — you always get a good meal. Restaurant B sometimes serves a $50-quality meal but sometimes a $5-quality disaster. Even if their *average* meal quality is the same, most people prefer Restaurant A. The Sharpe Ratio captures this preference — it rewards consistency.

**Key insight:** A high-return but wildly volatile portfolio can have a *lower* Sharpe Ratio than a moderate-return but steady portfolio. MPT optimization maximizes this ratio to find the best risk-adjusted allocation.

---

**References:**
1. [Sharpe Ratio — Investopedia](https://www.investopedia.com/terms/s/sharperatio.asp) — Comprehensive definition, formula, and examples
2. [William F. Sharpe, "The Sharpe Ratio" (1994)](https://web.stanford.edu/~wfsharpe/art/sr/SR.htm) — Original paper by the creator of the ratio
3. [Sharpe Ratio — CFA Institute](https://www.cfainstitute.org/en/membership/professional-development/refresher-readings/portfolio-risk-and-return-part-i) — CFA curriculum on portfolio risk and return
4. [Risk-Adjusted Return — Corporate Finance Institute](https://corporatefinanceinstitute.com/resources/career-map/sell-side/capital-markets/sharpe-ratio-definition-formula/) — Sharpe Ratio definition with worked examples
5. [Modern Portfolio Theory — Nobel Prize](https://www.nobelprize.org/prizes/economic-sciences/1990/sharpe/facts/) — William Sharpe's 1990 Nobel Prize in Economics

## 7. Optimize: Maximum Sharpe Ratio Portfolio

The **tangency portfolio** — highest return per unit of risk:

$$\max_w \frac{w^T \mu - r_f}{\sqrt{w^T \Sigma w}}$$

subject to $\sum w_i = 1$, $0 \leq w_i \leq 0.25$ (long-only, capped).

In [36]:
# Max Sharpe Ratio optimization
ef_sharpe = EfficientFrontier(mu, cov_matrix, weight_bounds=(0, MAX_WEIGHT))
ef_sharpe.max_sharpe(risk_free_rate=RISK_FREE_RATE)

sharpe_weights = ef_sharpe.clean_weights(cutoff=0.01)
sharpe_perf = ef_sharpe.portfolio_performance(verbose=False, risk_free_rate=RISK_FREE_RATE)

print("=" * 80)
print("  MAX SHARPE RATIO PORTFOLIO")
print("=" * 80)
print(f"  Expected Return:  {sharpe_perf[0]:>8.2%}")
print(f"  Volatility:       {sharpe_perf[1]:>8.2%}")
print(f"  Sharpe Ratio:     {sharpe_perf[2]:>8.2f}")
print()
print(f"  {'Symbol':<8} {'Company':<30} {'Weight':>8}  {'Current':>8}  {'Change':>8}")
print(f"  {'-'*8} {'-'*30} {'-'*8}  {'-'*8}  {'-'*8}")

for sym, w in sorted(sharpe_weights.items(), key=lambda x: -x[1]):
    if w > 0:
        cur_w = current_weights.get(sym, 0)
        delta = w - cur_w
        name = sym_name.get(sym, "")[:30]
        print(f"  {sym:<8} {name:<30} {w:>7.1%}   {cur_w:>7.1%}   {delta:>+7.1%}")

n_assets = sum(1 for w in sharpe_weights.values() if w > 0)
print(f"\n  Concentrated in {n_assets} assets (from {len(opt_symbols)} available)")
print("=" * 80)

  MAX SHARPE RATIO PORTFOLIO
  Expected Return:    66.73%
  Volatility:         19.35%
  Sharpe Ratio:         3.22

  Symbol   Company                          Weight   Current    Change
  -------- ------------------------------ --------  --------  --------
  GE       GE AEROSPACE COM NEW             25.0%      0.2%    +24.8%
  WMT      WALMART INC COM                  25.0%      0.1%    +24.9%
  KO       COCA-COLA CO                     13.1%      0.1%    +13.1%
  PLTR     PALANTIR TECHNOLOGIES INC CL A   12.8%      0.4%    +12.3%
  NVDA     NVIDIA CORPORATION COM           11.2%      6.5%     +4.6%
  GOOGL    ALPHABET INC CAP STK CL A        10.3%      3.1%     +7.3%
  WM       WASTE MANAGEMENT INC              1.9%      0.9%     +1.0%

  Concentrated in 7 assets (from 79 available)


## 8. Optimize: Minimum Volatility Portfolio

The **leftmost point** on the efficient frontier — lowest possible risk:

$$\min_w \sqrt{w^T \Sigma w}$$

---

### What is the Minimum Volatility Portfolio?

The **Minimum Volatility (Min Vol) Portfolio** is the combination of assets that produces the **lowest possible overall risk** (standard deviation of returns) — regardless of expected return. On the efficient frontier chart, it sits at the **leftmost point** of the curve.

### Why does it matter?

> *"I don't care about maximizing returns — I just want the smoothest ride possible."*

This portfolio answers that question. It is the mathematically optimal way to **minimize the ups and downs** of your portfolio value, given the available assets and their historical co-movements.

### Intuitive Analogy — The Shock Absorber Portfolio

Imagine you're building a car suspension system:
- Each **asset** is a spring with its own stiffness (volatility)
- The **correlations** between assets are how the springs interact — some amplify bumps, others cancel them out
- The Min Vol portfolio finds the **exact combination of springs** that produces the smoothest ride over any road

Two assets that individually bounce a lot can **cancel each other's movement** if they're negatively correlated. The optimizer exploits this effect to find the combination with the least total bouncing.

### How it works — step by step

1. **Start with the covariance matrix** $\Sigma$ — this captures both individual asset volatilities AND how they move together (correlations)

2. **Define portfolio variance** as a quadratic function of weights:
$$\sigma_p^2 = w^T \Sigma w = \sum_{i}\sum_{j} w_i w_j \sigma_{ij}$$

3. **Minimize** this subject to constraints:
   - $\sum w_i = 1$ — weights must sum to 100% (fully invested)
   - $0 \leq w_i \leq 0.25$ — long-only, max 25% per asset (our cap)

4. **Solve** using quadratic programming (convex optimization — guaranteed global minimum)

### Key properties of the Min Vol Portfolio

| Property | Explanation |
|----------|-------------|
| **Lowest risk point** | No other feasible portfolio has lower volatility |
| **Diversified** | Typically holds many assets to exploit correlation benefits |
| **Lower return** | Usually earns less than Max Sharpe — you sacrifice return for stability |
| **Defensive** | Tends to overweight low-beta, low-correlation assets (utilities, staples, bonds) |
| **Robust** | Less sensitive to errors in expected return estimates (only uses covariance) |

### When to prefer Min Vol over Max Sharpe

- **Near retirement** — capital preservation matters more than growth
- **High uncertainty** — when you distrust return forecasts (Min Vol ignores expected returns in the objective)
- **Drawdown aversion** — if you can't stomach seeing -20% portfolio drops
- **Short time horizons** — less time to recover from volatility
- **Bear markets** — Min Vol portfolios historically outperform in downturns

### The diversification magic — why it works

The key insight is that portfolio volatility is **not** the weighted average of individual volatilities. Thanks to **imperfect correlations**, the portfolio's risk is always less than or equal to the weighted average:

$$\sigma_p \leq \sum_i w_i \sigma_i$$

Equality only holds when all assets are perfectly correlated ($\rho = 1$). In practice, correlations are usually well below 1, so the optimizer can find combinations where assets' movements partially cancel out.

**Example:** If Stock A rises when Stock B falls (negative correlation), holding both reduces total portfolio swings — even if each stock individually is very volatile.

### Limitations

- **Backward-looking** — assumes historical correlations will persist (they shift in crises)
- **Correlation breakdown** — in market crashes, correlations spike toward 1, reducing diversification benefits exactly when you need them most
- **Opportunity cost** — you may leave significant returns on the table
- **Estimation error** — covariance matrices for 80+ assets require many parameters and can be noisy (mitigated by Ledoit-Wolf shrinkage in Section 5)

---

**References:**
1. [Minimum Variance Portfolio — Investopedia](https://www.investopedia.com/terms/m/minimum-variance-portfolio.asp) — Definition, formula, and intuition
2. [Global Minimum Variance Portfolio — CFA Institute](https://www.cfainstitute.org/en/membership/professional-development/refresher-readings/portfolio-risk-and-return-part-i) — CFA Level I curriculum on portfolio construction
3. [Jagannathan & Ma, "Risk Reduction in Large Portfolios" (2003)](https://doi.org/10.1111/1540-6261.00580) — Journal of Finance paper showing min-variance portfolios outperform with constrained weights
4. [Clarke, de Silva & Thorley, "Minimum-Variance Portfolios in the U.S. Equity Market" (2006)](https://doi.org/10.3905/jpm.2006.661366) — Journal of Portfolio Management — empirical evidence that min-vol beats cap-weighted indices
5. [Ledoit & Wolf, "A Well-Conditioned Estimator for Large-Dimensional Covariance Matrices" (2004)](https://doi.org/10.1016/S0047-259X(03)00096-4) — The shrinkage estimator used in our covariance calculation
6. [Low Volatility Anomaly — AQR Capital](https://www.aqr.com/Insights/Research/Journal-Article/Betting-Against-Beta) — "Betting Against Beta" — why low-volatility stocks earn higher risk-adjusted returns than theory predicts

In [37]:
# Min Volatility optimization
ef_minvol = EfficientFrontier(mu, cov_matrix, weight_bounds=(0, MAX_WEIGHT))
ef_minvol.min_volatility()

minvol_weights = ef_minvol.clean_weights(cutoff=0.01)
minvol_perf = ef_minvol.portfolio_performance(verbose=False, risk_free_rate=RISK_FREE_RATE)

print("=" * 80)
print("  MINIMUM VOLATILITY PORTFOLIO")
print("=" * 80)
print(f"  Expected Return:  {minvol_perf[0]:>8.2%}")
print(f"  Volatility:       {minvol_perf[1]:>8.2%}")
print(f"  Sharpe Ratio:     {minvol_perf[2]:>8.2f}")
print()
print(f"  {'Symbol':<8} {'Company':<30} {'Weight':>8}")
print(f"  {'-'*8} {'-'*30} {'-'*8}")

for sym, w in sorted(minvol_weights.items(), key=lambda x: -x[1]):
    if w > 0:
        name = sym_name.get(sym, "")[:30]
        print(f"  {sym:<8} {name:<30} {w:>7.1%}")

n_assets = sum(1 for w in minvol_weights.values() if w > 0)
print(f"\n  Diversified across {n_assets} assets")
print("=" * 80)

  MINIMUM VOLATILITY PORTFOLIO
  Expected Return:    15.03%
  Volatility:         10.01%
  Sharpe Ratio:         1.05

  Symbol   Company                          Weight
  -------- ------------------------------ --------
  FBIFX    FIDELITY FREEDOM INDEX 2040 IN   20.6%
  KO       COCA-COLA CO                     19.3%
  WM       WASTE MANAGEMENT INC             12.1%
  VXUS     VANGUARD TOTAL INTERNATIONAL S    8.6%
  VZ       VERIZON COMMUNICATIONS INC        6.8%
  WMT      WALMART INC COM                   6.5%
  O        REALTY INCOME CORP COM            6.1%
  MSFT     MICROSOFT CORP                    6.1%
  WTRG     ESSENTIAL UTILS INC COM           3.7%
  GOOGL    ALPHABET INC CAP STK CL A         3.5%
  SCHD     SCHWAB US DIVIDEND EQUITY ETF     2.1%
  FANG     DIAMONDBACK ENERGY INC COM USD    1.8%
  NFLX     NETFLIX INC                       1.3%

  Diversified across 13 assets


## 9. Efficient Frontier Visualization

Plot the risk-return tradeoff curve with Current, Max Sharpe, and Min Volatility marked.

In [38]:
# Generate efficient frontier points using CLA
cla = CLA(mu, cov_matrix, weight_bounds=(0, MAX_WEIGHT))
(ef_returns, ef_vols, _) = cla.efficient_frontier(points=100)

# Create the plot
fig = go.Figure()

# Efficient frontier curve
fig.add_trace(go.Scatter(
    x=ef_vols, y=ef_returns,
    mode="lines",
    name="Efficient Frontier",
    line=dict(color="blue", width=2),
))

# Individual assets — labels show "SYMBOL (Company Name)"
asset_vols = np.sqrt(np.diag(cov_matrix))
asset_labels = [f"{s} ({sym_name.get(s, '')[:25]})" for s in mu.index]
fig.add_trace(go.Scatter(
    x=asset_vols, y=mu.values,
    mode="markers",
    name="Individual Assets",
    text=asset_labels,
    hovertemplate="%{text}<br>Vol: %{x:.1%}<br>Return: %{y:.1%}<extra></extra>",
    marker=dict(size=6, color="gray", opacity=0.6),
))

# Current portfolio
fig.add_trace(go.Scatter(
    x=[current_vol], y=[current_return],
    mode="markers+text",
    name=f"Current (Sharpe={current_sharpe:.2f})",
    text=["Current"],
    textposition="bottom right",
    marker=dict(size=14, color="red", symbol="star"),
))

# Max Sharpe portfolio
fig.add_trace(go.Scatter(
    x=[sharpe_perf[1]], y=[sharpe_perf[0]],
    mode="markers+text",
    name=f"Max Sharpe ({sharpe_perf[2]:.2f})",
    text=["Max Sharpe"],
    textposition="top left",
    marker=dict(size=14, color="green", symbol="diamond"),
))

# Min Volatility portfolio
fig.add_trace(go.Scatter(
    x=[minvol_perf[1]], y=[minvol_perf[0]],
    mode="markers+text",
    name=f"Min Vol (σ={minvol_perf[1]:.1%})",
    text=["Min Vol"],
    textposition="bottom left",
    marker=dict(size=14, color="orange", symbol="square"),
))

fig.update_layout(
    title="Efficient Frontier — My Portfolio",
    xaxis_title="Annual Volatility (Risk)",
    yaxis_title="Expected Annual Return",
    width=950, height=600,
    legend=dict(x=0.02, y=0.98),
    xaxis=dict(tickformat=".0%"),
    yaxis=dict(tickformat=".0%"),
)
fig.show()

TypeError: only 0-dimensional arrays can be converted to Python scalars

## 10. Portfolio Comparison Summary

Side-by-side comparison of all three portfolios.

In [ ]:
comparison = pd.DataFrame({
    "Current": [current_return, current_vol, current_sharpe],
    "Max Sharpe": [sharpe_perf[0], sharpe_perf[1], sharpe_perf[2]],
    "Min Volatility": [minvol_perf[0], minvol_perf[1], minvol_perf[2]],
}, index=["Expected Return", "Volatility", "Sharpe Ratio"])

print("\n" + "=" * 60)
print("  PORTFOLIO COMPARISON")
print("=" * 60)
print(comparison.to_string(float_format=lambda x: f"{x:.4f}"))
print("=" * 60)

# Bar chart comparison
fig = go.Figure()
for col in comparison.columns:
    fig.add_trace(go.Bar(name=col, x=comparison.index, y=comparison[col].values))

fig.update_layout(
    barmode="group",
    title="Portfolio Comparison: Current vs Optimized",
    width=800, height=450,
    yaxis_title="Value",
)
fig.show()

## 11. Weight Comparison — Rebalancing Direction

Diverging bar chart: which positions should **grow** (buy) and which should **shrink** (sell)
to move from current allocation to Max Sharpe optimal.

In [ ]:
# Build comparison DataFrame
weight_comp = pd.DataFrame({
    "Current": current_weights,
    "Max Sharpe": pd.Series(sharpe_weights),
}).fillna(0)

weight_comp["Delta"] = weight_comp["Max Sharpe"] - weight_comp["Current"]
weight_comp = weight_comp[weight_comp.abs().max(axis=1) > 0.005]  # filter noise
weight_comp = weight_comp.sort_values("Delta")

# Add company names as labels: "SYMBOL — Company Name"
weight_labels = [f"{s} — {sym_name.get(s, '')[:25]}" for s in weight_comp.index]

# Diverging bar chart
fig = go.Figure()
colors = ["green" if d > 0 else "red" for d in weight_comp["Delta"]]
fig.add_trace(go.Bar(
    y=weight_labels,
    x=weight_comp["Delta"],
    orientation="h",
    marker_color=colors,
    text=[f"{d:+.1%}" for d in weight_comp["Delta"]],
    textposition="outside",
))

fig.update_layout(
    title="Rebalancing Direction: Current → Max Sharpe",
    xaxis_title="Weight Change",
    xaxis=dict(tickformat=".0%"),
    width=900, height=max(400, len(weight_comp) * 24),
    margin=dict(l=250),
)
fig.show()

## 12. Discrete Allocation (Whole Shares)

Convert fractional optimal weights into a concrete **trade plan** using
`pypfopt.DiscreteAllocation` with a greedy algorithm.

In [ ]:
# Latest prices for discrete allocation
latest_prices = get_latest_prices(prices)

# Total value of symbols in optimization universe
opt_total_value = portfolio.loc[portfolio.index.isin(opt_symbols), "total_value"].sum()

# Discrete allocation for Max Sharpe
da = DiscreteAllocation(
    sharpe_weights, latest_prices, total_portfolio_value=opt_total_value
)
allocation, leftover = da.greedy_portfolio()

print("=" * 90)
print(f"  DISCRETE ALLOCATION (Max Sharpe) — ${opt_total_value:,.0f} portfolio")
print("=" * 90)
print(f"  {'Symbol':<8} {'Company':<25} {'Shares':>8} {'Price':>10} {'Value':>12} {'Weight':>8}")
print(f"  {'-'*8} {'-'*25} {'-'*8} {'-'*10} {'-'*12} {'-'*8}")

alloc_total = 0
for sym, shares in sorted(allocation.items(), key=lambda x: -x[1] * latest_prices[x[0]]):
    price = latest_prices[sym]
    value = shares * price
    weight = value / opt_total_value
    alloc_total += value
    name = sym_name.get(sym, "")[:25]
    print(f"  {sym:<8} {name:<25} {shares:>8} ${price:>9,.2f} ${value:>11,.2f} {weight:>7.1%}")

print(f"\n  Allocated: ${alloc_total:,.2f}")
print(f"  Leftover:  ${leftover:,.2f}")
print("=" * 90)

## 13. Per-Account Breakdown

Show the current allocation within each Fidelity account separately.

| Account Type | Rebalancing Impact |
|-------------|--------------------|
| **Taxable (Individual TOD)** | Triggers capital gains/losses |
| **Roth IRA / 401k** | Rebalancing is tax-free |
| **529 / HSA** | Limited options, separate optimization |

In [ ]:
# Per-account summary using service layer
account_alloc = service.allocation_by_account(positions)
print("=" * 90)
print("  PER-ACCOUNT ALLOCATION")
print("=" * 90)
display(account_alloc)

# Sunburst chart: Account → Symbol
sunburst_df = positions[["account_name", "symbol", "current_value"]].copy()
sunburst_df = sunburst_df[sunburst_df["current_value"] > 0]

fig = px.sunburst(
    sunburst_df,
    path=["account_name", "symbol"],
    values="current_value",
    title="Portfolio Structure: Account \u2192 Symbol",
    width=800, height=700,
)
fig.show()

## 14. Risk Decomposition — Correlation Heatmap

Correlation matrix of daily returns for the top holdings.  
High correlations reduce diversification benefits.

In [ ]:
# Daily returns
daily_returns = prices.pct_change().dropna()

# Correlation for top N holdings
top_n = min(20, len(opt_symbols))
top_symbols = portfolio.index[:top_n].tolist()
top_in_prices = [s for s in top_symbols if s in daily_returns.columns]

corr = daily_returns[top_in_prices].corr()

# Use "SYMBOL (Company)" as axis labels
corr_labels = [f"{s} ({sym_name.get(s, '')[:20]})" for s in top_in_prices]
corr_display = corr.copy()
corr_display.index = corr_labels
corr_display.columns = corr_labels

fig = px.imshow(
    corr_display,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title=f"Return Correlation Matrix (Top {len(top_in_prices)} Holdings)",
    width=900, height=800,
)
fig.show()

# Average pairwise correlation
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
avg_corr = corr.where(mask).stack().mean()
print(f"\nAverage pairwise correlation (top {len(top_in_prices)}): {avg_corr:.3f}")

## 15. Cumulative Returns Backtest

Compare buy-and-hold of the **current** portfolio vs the **Max Sharpe optimal**
over the historical period.

In [ ]:
# Cumulative returns for current vs optimal
current_w = current_weights.reindex(daily_returns.columns).fillna(0)
sharpe_w = pd.Series(sharpe_weights).reindex(daily_returns.columns).fillna(0)

current_port_returns = daily_returns @ current_w
optimal_port_returns = daily_returns @ sharpe_w

cumulative = pd.DataFrame({
    "Current Portfolio": (1 + current_port_returns).cumprod() - 1,
    "Max Sharpe (Optimal)": (1 + optimal_port_returns).cumprod() - 1,
})

fig = px.line(
    cumulative,
    title="Cumulative Returns: Current vs. Optimal Portfolio",
    labels={"value": "Cumulative Return", "variable": "Portfolio"},
    width=950, height=500,
)
fig.update_layout(
    yaxis=dict(tickformat=".0%"),
    legend=dict(x=0.02, y=0.98),
)
fig.show()

# Summary stats
print("\nBacktest Summary (full period):")
for col in cumulative.columns:
    total_ret = cumulative[col].iloc[-1]
    ann_ret = (1 + total_ret) ** (252 / len(cumulative)) - 1
    daily_vol = cumulative[col].pct_change().std()
    print(f"  {col}: {total_ret:+.1%} cumulative, {ann_ret:.1%} annualized")

## 16. Key Takeaways & Next Steps

**Interpretation Guide:**
- If your current Sharpe ratio is **close to the Max Sharpe** portfolio, your allocation is near-optimal
- If the efficient frontier shows your position **well below the curve**, there is significant room for improvement
- The **discrete allocation** gives a concrete trade plan (buy/sell whole shares)

**Caveats:**
- MPT assumes returns are normally distributed and past correlations persist — both are approximations
- Transaction costs, taxes, and wash-sale rules are NOT modeled
- 25% per-asset cap prevents extreme concentration but may limit returns

**Future Work** (see `portfolio_optimization_strategy.md`):
1. **Black-Litterman model** — incorporate subjective views into expected returns
2. **Tax-aware rebalancing** — minimize taxable events in Individual account
3. **Sector constraints** — ensure diversification across GICS sectors
4. **Dividend optimization** — incorporate dividend yield into expected returns
5. **Monte Carlo simulation** — confidence intervals around expected returns
6. **Risk parity** — equal risk contribution from each asset
7. **Regime detection** — adjust covariance estimates for bull/bear markets
8. **Automated scheduling** — monthly re-optimization with drift tracking

In [ ]:
print("\u2501" * 60)
print("  OPTIMIZATION COMPLETE")
print("\u2501" * 60)
print(f"  Symbols analyzed:    {len(opt_symbols)}")
print(f"  Price history:       {prices.shape[0]} days")
print(f"  Portfolio value:     ${total_portfolio_value:,.0f}")
print(f"  Current Sharpe:      {current_sharpe:.2f}")
print(f"  Max Sharpe:          {sharpe_perf[2]:.2f}")
print(f"  Min Vol Sharpe:      {minvol_perf[2]:.2f}")
print(f"  Improvement potential: {sharpe_perf[2] - current_sharpe:+.2f} Sharpe points")
print("\u2501" * 60)